In [9]:
import pandas as pd

THR_OBS  = 0.5   # mm — wet-day threshold (Haylock 2008)
THR_PRED = 0.4   # mm — cutoff for predicted

preds = pd.read_parquet('../../../results/bayesnf/experiments/time_seasonality/WY_h1_10/preds.parquet')

# binary labels: True = dry
obs_dry  = preds['observed_mm'] < THR_OBS
pred_dry = preds['mean_mm']     < THR_PRED

# confusion matrix (dry-class perspective)
tn = ( obs_dry &  pred_dry).sum()  # correctly predicted dry
fp = ( obs_dry & ~pred_dry).sum()  # observed dry, predicted wet
fn = (~obs_dry &  pred_dry).sum()  # observed wet, predicted dry (miss)
tp = (~obs_dry & ~pred_dry).sum()  # correctly predicted wet

n      = len(preds)
n_dry  = obs_dry.sum()
n_wet  = (~obs_dry).sum()

dry_recall = tn / n_dry
print(f'Correctly predicted dry: {tn:,} / {n_dry:,} = {dry_recall:.1%}')

print(f'\nConfusion matrix:')
print(f'                       pred dry (<{THR_PRED})   pred wet (≥{THR_PRED})')
print(f'  obs dry (<{THR_OBS} mm)   {tn:>10,}        {fp:>10,}')
print(f'  obs wet (≥{THR_OBS} mm)   {fn:>10,}        {tp:>10,}')

print(f'\n--- per-class metrics ---')
print(f'  Dry recall     (TN-rate):  {tn/n_dry:.3f}')
print(f'  Wet recall     (TP-rate):  {tp/n_wet:.3f}')
print(f'  Dry precision  :           {tn/(tn+fn):.3f}')
print(f'  Wet precision  :           {tp/(tp+fp):.3f}')
print(f'  Overall accuracy:          {(tn+tp)/n:.3f}')

# standard precipitation verification scores
pod  = tp / (tp + fn)              # probability of detection (wet)
far  = fp / (fp + tp) if (fp+tp) else 0   # false alarm ratio
csi  = tp / (tp + fp + fn)         # critical success index (= Jaccard for wet)

# HSS — Heidke Skill Score (random-baseline corrected)
expected_correct = ((tp+fp)*(tp+fn) + (fn+tn)*(fp+tn)) / n
hss = (tp + tn - expected_correct) / (n - expected_correct)

print(f'\n--- precipitation verification scores ---')
print(f'  POD  (wet hit rate):       {pod:.3f}')
print(f'  FAR  (wet false alarm):    {far:.3f}')
print(f'  CSI  (critical success):   {csi:.3f}')
print(f'  HSS  (Heidke skill):       {hss:.3f}   [>0 = better than random]')

Correctly predicted dry: 457,361 / 548,637 = 83.4%

Confusion matrix:
                       pred dry (<0.4)   pred wet (≥0.4)
  obs dry (<0.5 mm)      457,361            91,276
  obs wet (≥0.5 mm)        9,776           304,841

--- per-class metrics ---
  Dry recall     (TN-rate):  0.834
  Wet recall     (TP-rate):  0.969
  Dry precision  :           0.979
  Wet precision  :           0.770
  Overall accuracy:          0.883

--- precipitation verification scores ---
  POD  (wet hit rate):       0.969
  FAR  (wet false alarm):    0.230
  CSI  (critical success):   0.751
  HSS  (Heidke skill):       0.761   [>0 = better than random]
